# Backgammon: the doubling cube, and dice odds

The doubling cube turns backgammon from a race into a betting game. It is a six-sided die marked 2, 4, 8, 16, 32, 64. At the start it sits in the middle at value 1, owned by no one. Before your roll you may **double**: offer to play on for twice the stakes. Your opponent must then **take** — accept, after which the cube (now showing 2) sits on their side and only they may double next — or **drop**, conceding the current stake and ending the game.

Owning the cube is valuable: it means only you can raise the stakes next. So handing it over is a real decision, and accepting it turns on a single number — your chance of winning from here.

## When to double, take, or drop

Play a money game (unlimited, no match) and, for a moment, ignore gammons and suppose the cube is *dead* — no more doubling after this one. You are offered the cube at value 1, raising the stake to 2.

- **Drop:** you lose 1 point, for certain.
- **Take:** you play on for 2 points. Winning with probability $p$, your equity is $2p - 2(1 - p) = 4p - 2$.

Taking beats dropping when $4p - 2 \ge -1$, that is $p \ge \tfrac{1}{4}$. This is the famous **25% take point**: accept any double you win at least a quarter of the time. With a *live* cube — you can redouble later — cube ownership is worth extra and the take point falls to about **20%**.

From the doubler's side, the mirror of the take point is the **cash point**: once the opponent's chances fall below their take point (you are winning about 80%), doubling them out is right — they must pass and you bank a point. Between roughly 50% and that cash point lies the **doubling window**: good enough to double, not so good they must drop.

One more region: if you are winning *and* threatening a **gammon** (a double-stake win), doubling can be a mistake — the opponent simply drops and you collect one point when playing on might have won two. That is **too good to double**. Gammons pull every threshold around; a rough money rule folds them in through $m = W + G/2$, with $W$ your total win chance and $G$ your gammon-win chance: double past $m = 0.5$, opponent drops past $m = 0.75$, too good past $m = 1$.

## The money-game cube map

Each cell is a position: your win chance down the side, your gammon-win chance across the top (a gammon is a win too, so it never exceeds the win chance). The colour is the correct cube action in an unlimited game, from the $m = W + G/2$ rule.

- **Yellow** — double, and the opponent has a **take**.
- **Blue** — double, and the opponent must **drop** (you cash a point).

Grey is *no double* (too weak, along the bottom) or *too good* (top-right — play on for the gammon).

In [1]:
import plotly.graph_objects as go

WINS = list(range(50, 100, 5))
GAMS = list(range(0, 46, 5))


def action(win, gammon):
    w, g = win / 100, gammon / 100
    if g > w:
        return None
    m = w + g / 2
    if m < 0.5:
        return 0
    if m <= 0.75:
        return 1
    if m <= 1.0:
        return 2
    return 3


LABELS = ["no double", "double / take", "double / drop", "too good"]
CELL = {0: "", 1: "take", 2: "drop", 3: "too good", None: ""}
z = [[action(w, g) for g in GAMS] for w in WINS]
text = [[CELL[action(w, g)] for g in GAMS] for w in WINS]

colors = ["#eef0f2", "#ffd94d", "#4da3ff", "#c2c2c2"]
scale = []
for i, c in enumerate(colors):
    scale += [[i / 4, c], [(i + 1) / 4, c]]

fig = go.Figure(go.Heatmap(
    z=z, x=[f"{g}%" for g in GAMS], y=[f"{w}%" for w in WINS],
    text=text, texttemplate="%{text}", textfont=dict(size=11),
    colorscale=scale, zmin=-0.5, zmax=3.5, xgap=1, ygap=1,
    colorbar=dict(tickvals=[0, 1, 2, 3], ticktext=LABELS, title="action")))
fig.update_layout(
    title="Money game: double / take / drop by win and gammon chance",
    xaxis_title="gammon-win chance", yaxis_title="game-win chance",
    margin=dict(t=60))
fig.show()

## Match play: the score changes everything

In a match to a set number of points, a win is not worth a fixed amount — it is worth how much closer it brings you to the finish. So the take point stops being a flat 25% and depends on **both scores**, read through a *match equity table* (MET): the chance of winning the match from a given score. The map below uses the classic Woolsey–Heinrich MET.

Scores are **points away** — how many each player still needs. When the doubler ($d$-away) turns the cube to 2, the taker ($t$-away) compares three match equities:

$$\text{TP} = \frac{a - b}{c - b}, \qquad \begin{aligned} a &= \text{equity after dropping (doubler scores 1)} \\ b &= \text{equity after taking and losing 2} \\ c &= \text{equity after taking and winning 2} \end{aligned}$$

Drag the two sliders to set the giver's and taker's points-away; the crosshair marks that cell and the number in it is the taker's take point. Landmarks fall straight out: **2-away/2-away is about 30%** — after the take the cube is dead (2 points wins the match), so no redouble value sweetens the take; the **trailer at 2-away/4-away needs ~34%**, while the **leader there takes freely at ~20%**. At **double match point** (1-away/1-away, off the grid) the take point is effectively 0% — dropping hands over the match — so you always take, then need 50% to win.

These take points are cubeless — no future redoubles — so for live-cube scores like 4-away/4-away they run slightly high; a redouble you still own lowers them.

In [2]:
import plotly.graph_objects as go

MET_PCT = [
    [50, 70, 75, 83, 85, 90, 91, 94, 95, 97, 97, 98, 98, 99, 99],
    [30, 50, 60, 68, 75, 81, 85, 88, 91, 93, 94, 95, 96, 97, 98],
    [25, 40, 50, 59, 66, 71, 76, 80, 84, 87, 90, 92, 94, 95, 96],
    [17, 32, 41, 50, 58, 64, 70, 75, 79, 83, 86, 88, 90, 92, 93],
    [15, 25, 34, 42, 50, 57, 63, 68, 73, 77, 81, 84, 87, 89, 90],
    [10, 19, 29, 36, 43, 50, 56, 62, 67, 72, 76, 79, 82, 85, 87],
    [9, 15, 24, 30, 37, 44, 50, 56, 61, 66, 70, 74, 78, 81, 84],
    [6, 12, 20, 25, 32, 38, 44, 50, 55, 60, 65, 69, 73, 77, 80],
    [5, 9, 16, 21, 27, 33, 39, 45, 50, 55, 60, 64, 68, 72, 76],
    [3, 7, 13, 17, 23, 28, 34, 40, 45, 50, 55, 60, 64, 68, 71],
    [3, 6, 10, 14, 19, 24, 30, 35, 40, 45, 50, 55, 59, 63, 67],
    [2, 5, 8, 12, 16, 21, 26, 31, 36, 40, 45, 50, 54, 58, 62],
    [2, 4, 6, 10, 13, 18, 22, 27, 32, 36, 41, 46, 50, 54, 58],
    [1, 3, 5, 8, 11, 15, 19, 23, 28, 32, 37, 42, 46, 50, 54],
    [1, 2, 4, 7, 10, 13, 16, 20, 24, 29, 33, 38, 42, 46, 50],
]


def met(t, d):
    if t <= 0:
        return 100.0
    if d <= 0:
        return 0.0
    return float(MET_PCT[t - 1][d - 1])


def take_point(taker_away, giver_away):
    a = met(taker_away, giver_away - 1)
    b = met(taker_away, giver_away - 2)
    c = met(taker_away - 2, giver_away)
    if c == b:
        return None
    return (a - b) / (c - b) * 100


aways = list(range(2, 10))
n = len(aways)
z = [[take_point(t, d) for t in aways] for d in aways]
text = [[f"{v:.0f}" if v is not None else "" for v in row] for row in z]

fig = go.Figure(go.Heatmap(
    z=z, x=[str(t) for t in aways], y=[str(d) for d in aways],
    text=text, texttemplate="%{text}%", textfont=dict(size=11),
    colorscale="YlGnBu", xgap=1, ygap=1, colorbar=dict(title="take point")))

lo, hi = -0.5, n - 0.5
fig.update_layout(shapes=[
    dict(type="rect", xref="x", yref="y", x0=lo, x1=hi, y0=lo, y1=0.5,
         fillcolor="rgba(0,0,0,0.20)", line=dict(width=0)),
    dict(type="rect", xref="x", yref="y", x0=lo, x1=0.5, y0=lo, y1=hi,
         fillcolor="rgba(0,0,0,0.20)", line=dict(width=0)),
])

giver_steps = [dict(method="relayout", label=str(d),
                    args=[{"shapes[0].y0": i - 0.5, "shapes[0].y1": i + 0.5}])
               for i, d in enumerate(aways)]
taker_steps = [dict(method="relayout", label=str(t),
                    args=[{"shapes[1].x0": j - 0.5, "shapes[1].x1": j + 0.5}])
               for j, t in enumerate(aways)]

fig.update_layout(
    sliders=[
        dict(active=0, y=-0.02, len=0.6, currentvalue={"prefix": "cube giver away = "},
             steps=giver_steps),
        dict(active=0, y=-0.32, len=0.6, currentvalue={"prefix": "cube taker away = "},
             steps=taker_steps),
    ],
    title="Match-play take point by score (Woolsey MET)",
    xaxis_title="cube taker — points away",
    yaxis_title="cube giver — points away",
    margin=dict(t=60, b=140))
fig.show()

## Dice odds you can do in your head

Every turn is one of 36 equally likely ordered rolls ($6 \times 6$). A few counts cover most decisions.

- **A direct shot** — hitting a blot 1 to 6 pips away — needs one die to show that number: **11 of 36** rolls contain a given number, just over 30%.
- **Combined numbers** — reaching a point 7 to 12 pips off with both dice or a double — thin out fast: 7 has 6 shots, 8 has 6, 9 has 5, 10 has 3, 11 has 2, 12 has 3. Past 12 only doubles reach: 15, 16, 18, 20, 24 are one shot each.
- **Entering from the bar** against a home board with $n$ points made: you fail only if *both* dice show a closed point, so $P(\text{enter}) = 1 - (n/6)^2$. Three points closed still lets you in 75% of the time; six is a closeout.
- **Doubles** come up **1 in 6**. **Not** rolling a given number is $(5/6)^2 = 25/36$, about 69%. The average roll moves **8.17 pips** (doubles count four dice).

The helpers below derive these by enumerating all 36 rolls, so you can check any distance or board yourself.

In [3]:
DICE = [(a, b) for a in range(1, 7) for b in range(1, 7)]


def reachable(a, b):
    return {a, 2 * a, 3 * a, 4 * a} if a == b else {a, b, a + b}


def shots(distance):
    return sum(1 for a, b in DICE if distance in reachable(a, b))


def hit_prob(distance):
    return shots(distance) / 36


def enter_prob(points_closed):
    return 1 - (points_closed / 6) ** 2


for k in range(1, 13):
    print(f"hit {k:>2} pips away: {shots(k):>2}/36 = {hit_prob(k) * 100:4.1f}%")
print()
for closed in range(1, 7):
    print(f"enter vs {closed} closed board points: {enter_prob(closed) * 100:5.1f}%")

hit  1 pips away: 11/36 = 30.6%
hit  2 pips away: 12/36 = 33.3%
hit  3 pips away: 14/36 = 38.9%
hit  4 pips away: 15/36 = 41.7%
hit  5 pips away: 15/36 = 41.7%
hit  6 pips away: 17/36 = 47.2%
hit  7 pips away:  6/36 = 16.7%
hit  8 pips away:  6/36 = 16.7%
hit  9 pips away:  5/36 = 13.9%
hit 10 pips away:  3/36 =  8.3%
hit 11 pips away:  2/36 =  5.6%
hit 12 pips away:  3/36 =  8.3%

enter vs 1 closed board points:  97.2%
enter vs 2 closed board points:  88.9%
enter vs 3 closed board points:  75.0%
enter vs 4 closed board points:  55.6%
enter vs 5 closed board points:  30.6%
enter vs 6 closed board points:   0.0%


**[In-browser notebook →](/lite/notebooks/index.html?path=2026-09-24-backgammon-cube-and-dice.ipynb)**